# M2: baseline ResNet-18 pe PatchCamelyon (Colab)

**Înainte:** urcă folderul proiectului în Google Drive ca `MyDrive/DSM_PCam/`, cu cel puțin `pcam/` și `train.py`.
Apoi *Runtime → Change runtime type → GPU* și rulează celulele în ordine.

- Codul se copiază din Drive pe discul local Colab, iar datele se descarcă tot local (~5 min).
- Rezultatele (`runs/`) se scriu direct în Drive, ca să nu se piardă la deconectare.

## 1. GPU

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "Nu ai GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

/bin/bash: line 1: nvidia-smi: command not found


AssertionError: Nu ai GPU: Runtime -> Change runtime type -> GPU

## 2. Drive și cod

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
PROJECT = '/content/drive/MyDrive/DSM_PCam'   # schimbă dacă l-ai pus în alt loc
WORK = '/content/work'
RUNS = f'{PROJECT}/runs'

assert os.path.exists(f'{PROJECT}/train.py') and os.path.isdir(f'{PROJECT}/pcam'), f'Nu găsesc codul în {PROJECT}'
os.makedirs(WORK, exist_ok=True)
shutil.copy(f'{PROJECT}/train.py', WORK)
shutil.copytree(f'{PROJECT}/pcam', f'{WORK}/pcam', dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('__pycache__'))
os.makedirs(RUNS, exist_ok=True)
%cd {WORK}
!ls -R pcam train.py

## 3. Date PCam
Aceleași fișiere ca pe Zenodo (record 2546921), de pe mirror-ul Google Drive folosit de `torchvision.datasets.PCAM`. Fiecare fișier e verificat cu MD5, apoi dezarhivat.

In [ ]:
!pip install -q gdown h5py
import gdown, hashlib, gzip, os, shutil

DATA = f'{WORK}/data'
os.makedirs(DATA, exist_ok=True)
FILES = {  # nume: (Google Drive id, md5 al .gz)
    'camelyonpatch_level_2_split_train_y.h5': ('1269yhu3pZDP8UYFQs-NYs3FPwuK-nGSG', '35c2d7259d906cfc8143347bb8e05be7'),
    'camelyonpatch_level_2_split_valid_y.h5': ('1bH8ZRbhSVAhScTS0p9-ZzGnX91cHT3uO', '2b85f58b927af9964a4c15b8f7e8f179'),
    'camelyonpatch_level_2_split_test_y.h5':  ('17BHrSrwWKjYsOgTMmoqrIjDy6Fa2o_gP', '60a7035772fbdb7f34eb86d4420cf66a'),
    'camelyonpatch_level_2_split_valid_x.h5': ('1hgshYGWK8V-eGRy8LToWJJgDU_rXWVJ3', 'd5b63470df7cfa627aeec8b9dc0c066e'),
    'camelyonpatch_level_2_split_test_x.h5':  ('1qV65ZqZvWzuIVthK8eVDhIwrbnsJdbg_', 'd8c2d60d490dbd479f8199bdfa0cf6ec'),
    'camelyonpatch_level_2_split_train_x.h5': ('1Ka0XfEMiwgCYPdTI-vv6eUElOBnKFKQ2', '1571f514728f59376b705fc836ff4b63'),
}

def md5(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

for name, (fid, want) in FILES.items():
    h5, gz = f'{DATA}/{name}', f'{DATA}/{name}.gz'
    if os.path.exists(h5):
        print('[skip]', name); continue
    for attempt in range(5):
        gdown.download(id=fid, output=gz, quiet=True, resume=True)
        if md5(gz) == want:
            break
        print(f'[warn] md5 greșit pentru {name}, reîncerc'); os.remove(gz)
    else:
        raise RuntimeError(f'Nu am reușit să descarc {name}')
    with gzip.open(gz, 'rb') as fi, open(h5, 'wb') as fo:
        shutil.copyfileobj(fi, fo, 1 << 24)
    os.remove(gz)
    print('[ ok ]', name, f'{os.path.getsize(h5) / 1e9:.2f} GB')

In [ ]:
import h5py
for s, n in [('train', 262144), ('valid', 32768), ('test', 32768)]:
    with h5py.File(f'{DATA}/camelyonpatch_level_2_split_{s}_x.h5') as fx, h5py.File(f'{DATA}/camelyonpatch_level_2_split_{s}_y.h5') as fy:
        assert fx['x'].shape == (n, 96, 96, 3) and fy['y'].shape == (n, 1, 1, 1), s
        print(s, fx['x'].shape, 'pozitive:', fy['y'][:].mean().round(3))

## 4. Antrenare baseline

ResNet-18 preantrenat pe ImageNet, augmentare geometrică (rot90 + flip), 100% din etichete, 10 epoci, AdamW lr 3e-4 cu cosine.
Checkpoint-ul se alege după AUC pe **validare**; setul de test e evaluat **o singură dată**, la final.
`--amp` folosește bf16 pe L4/A100 și fp16 + GradScaler pe T4.

In [ ]:
RUN = 'm2_baseline'
!python -u train.py --name {RUN} --amp --out {RUNS} 2>&1 | tee {RUNS}/{RUN}.log

## 5. Rezultate

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

out = f'{RUNS}/{RUN}'
hist = json.load(open(f'{out}/history.json'))
res = json.load(open(f'{out}/test_metrics.json'))
t = res['test']
print(f"best epoch {res['best_epoch']} | val AUC {res['best_val_auc']:.4f}")
print('TEST  ' + '  '.join(f'{k} {t[k]:.4f}' for k in ['auc', 'accuracy', 'precision', 'recall', 'specificity', 'f1']))
print(f"confusion: TP {t['tp']}  FN {t['fn']}  FP {t['fp']}  TN {t['tn']}")

ep = [h['epoch'] for h in hist]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ep, [h['train_loss'] for h in hist], 'o-', label='train')
ax[0].plot(ep, [h['val_loss'] for h in hist], 'o-', label='valid')
ax[0].set(title='Loss (BCE)', xlabel='epoch'); ax[0].legend()
ax[1].plot(ep, [h['val_auc'] for h in hist], 'o-', label='AUC')
ax[1].plot(ep, [h['val_recall'] for h in hist], 'o-', label='recall')
ax[1].axvline(res['best_epoch'], ls='--', c='gray', lw=1)
ax[1].set(title='Validare', xlabel='epoch'); ax[1].legend()
d = np.load(f'{out}/test_preds.npz')
fpr, tpr, _ = roc_curve(d['y'], d['p'])
ax[2].plot(fpr, tpr, label=f"AUC {t['auc']:.3f}"); ax[2].plot([0, 1], [0, 1], ':', c='gray')
ax[2].set(title='ROC pe test', xlabel='FPR (1 - specificity)', ylabel='TPR (recall)'); ax[2].legend()
plt.tight_layout(); plt.savefig(f'{out}/m2_curves.png', dpi=150); plt.show()